<a href="https://colab.research.google.com/github/Natalie-Bojorquez/CIS3920Group4/blob/main/CIS3920_Group4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [3920] Career Change Prediction
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-3920/refs/heads/main/data/career_change_prediction_100K.csv

## **Exploratory Data Analysis Notes**
*   def sample and standard pandas
*   pie chart over count bar graph
*   series.unique
*   lets try a Pearson correlation matrix? in the EDA file
nat to add in bar graphs from CIS 3120
*   important to look for missing values **if so you need to clean it
*   most visualizations to be done in matplotlib/seaborn, skip the pandas bcuz lame
*   most visualizations to be done in matplotlib/seaborn, skip the pandas bcuz lame
*   try using colormap if it works
*   once decide on what visualizations are the best! adjust to side by side for visualizations











In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 10/06/26 00:09:29


In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt     # same as 'import matplotlib.pyplot as plt'
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

### Review Raw File & Load Data into Pandas

In [21]:
# Read data file into a Pandas DataFrame
# CSV file format
# Field separator is ','
raw_df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-3920/refs/heads/main/data/career_change_prediction_100K.csv',
                 sep=',')

# 1) Data Sampling

In [22]:
# Display DataFrame dimensions
raw_df.shape  # 100,000 data instances (n rows), 25 features (m columns)

(100000, 25)

In [23]:
# Display DataFrame info: entries, columns, non-null counts, data types
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Employee ID                    100000 non-null  int64  
 1   Employer ID                    100000 non-null  int64  
 2   Field of Study                 100000 non-null  object 
 3   Current Occupation             100000 non-null  object 
 4   Age                            99965 non-null   float64
 5   Gender                         99791 non-null   object 
 6   Years of Experience            99514 non-null   float64
 7   Education Level                99696 non-null   object 
 8   Industry Growth Rate           99401 non-null   object 
 9   Job Satisfaction Level         99528 non-null   object 
 10  Work-Life Balance              99603 non-null   object 
 11  # Job Opportunities            99610 non-null   float64
 12  Salary                         

In [24]:
# Separate numeric and categorical variables
numeric_df = raw_df.select_dtypes(include=[np.number])
object_df = raw_df.select_dtypes(exclude=[np.number])

# Inspect categorical variables and their possible values
for col in object_df.columns:
  print(f'{col}: {object_df[col].unique()}', end = '\n''\n')


Field of Study: ['Biology' 'Economics' 'Medicine' 'Computer Science' 'Psychology'
 'Business' 'Education' 'Law' 'Mechanical Engineering' 'Arts']

Current Occupation: ['Economist' 'Doctor' 'Psychologist' 'Lawyer' 'Biologist' 'Teacher'
 'Artist' 'Business Analyst' 'Software Developer' 'Mechanical Engineer']

Gender: ['Female' 'Male' nan]

Education Level: ["Bachelor's" "Master's" 'PhD' 'High School' nan]

Industry Growth Rate: ['Medium' 'High' 'Low' nan]

Job Satisfaction Level: ['Low' 'Moderate' 'High' nan]

Work-Life Balance: ['Excellent' 'Moderate' 'Poor' 'Good' nan]

Job Security Level: ['Excellent' 'Moderate' 'Poor']

Interested in Career Change: ['No' 'Yes']

Skills Gap: ['High' 'Low' 'Moderate']

Family Influence Level: ['High' 'Low' nan 'Medium']

Has Mentor: ['N' 'Y' nan]

Has Certification: ['N' 'Y' nan]

Has Freelancing Experience: ['N' 'Y']

Willing to Relocate: ['No' 'Yes']

Professional Network Strength: ['Strong' 'Weak' 'Average']

Technical Proficiency Level: ['Intermedia

In [77]:
# Display summary statistics of numeric columns
raw_df.describe()

,Employee ID,Employer ID,Age,Years of Experience,# Job Opportunities,Salary,# Career Change Events
count,100000.000000,100000.000000,99965.000000,99514.000000,99610.000000,99653.000000,100000.000000
mean,499818.735850,53540.588850,39.602661,19.486324,50.283656,109003.983563,1.008990
std,231006.203232,24529.058757,11.623643,11.517150,28.874685,51212.416007,0.814688
min,100008.000000,13309.000000,20.000000,0.000000,1.000000,30005.000000,0.000000
25%,299737.750000,38672.000000,30.000000,10.000000,25.000000,61724.000000,0.000000
50%,499347.000000,53209.000000,40.000000,20.000000,50.000000,107684.000000,1.000000
75%,699583.750000,75617.000000,50.000000,29.000000,75.000000,153334.000000,2.000000
max,899992.000000,92445.000000,59.000000,39.000000,100.000000,199996.000000,2.000000


In [25]:
# Preview first 5 rows
raw_df.head()

,Employee ID,Employer ID,Field of Study,Current Occupation,Age,Gender,Years of Experience,Education Level,Industry Growth Rate,Job Satisfaction Level,...,Skills Gap,Family Influence Level,Has Mentor,Has Certification,Has Freelancing Experience,Willing to Relocate,Professional Network Strength,# Career Change Events,Technical Proficiency Level,Likely to Change Occupation
0,574511,38672,Biology,Economist,38.0,Female,7.0,Bachelor's,Medium,Low,...,High,High,N,N,N,No,Strong,1,Intermediate,Likely
1,447956,22363,Economics,Doctor,44.0,Female,0.0,Master's,Medium,Moderate,...,Low,Low,N,Y,N,No,Weak,1,Beginner,Likely
2,757898,42741,Medicine,Doctor,59.0,Male,10.0,PhD,Medium,Low,...,Low,NaN,N,Y,N,No,Average,2,Beginner,Likely
3,583528,92445,Medicine,Psychologist,57.0,Female,14.0,High School,Medium,Low,...,Moderate,Medium,N,N,N,No,Average,1,Expert,Likely
4,466234,39433,Computer Science,Lawyer,36.0,Female,23.0,PhD,High,Low,...,High,Low,N,N,N,Yes,Weak,0,Intermediate,Likely


In [26]:
# Preview last 5 rows
raw_df.tail()

,Employee ID,Employer ID,Field of Study,Current Occupation,Age,Gender,Years of Experience,Education Level,Industry Growth Rate,Job Satisfaction Level,...,Skills Gap,Family Influence Level,Has Mentor,Has Certification,Has Freelancing Experience,Willing to Relocate,Professional Network Strength,# Career Change Events,Technical Proficiency Level,Likely to Change Occupation
99995,168430,22363,Business,Business Analyst,20.0,Male,4.0,Bachelor's,Low,Low,...,High,Medium,Y,N,N,No,Average,0,Expert,Likely
99996,756046,92445,Psychology,Biologist,58.0,Male,9.0,PhD,Low,Low,...,Moderate,NaN,N,Y,N,Yes,Average,0,Intermediate,Likely
99997,624433,22363,Education,Biologist,27.0,Male,27.0,High School,Medium,Moderate,...,Low,Medium,N,N,N,No,Weak,1,Intermediate,Likely
99998,763689,18718,Business,Mechanical Engineer,23.0,Male,31.0,Bachelor's,Medium,High,...,High,NaN,Y,Y,Y,No,Weak,1,Intermediate,Unlikely
99999,506739,53209,Education,Psychologist,30.0,Male,34.0,PhD,Medium,High,...,Moderate,Medium,NaN,N,N,Yes,Average,0,Expert,Likely


In [27]:
# Display random sample of N rows
raw_df.sample(5)

,Employee ID,Employer ID,Field of Study,Current Occupation,Age,Gender,Years of Experience,Education Level,Industry Growth Rate,Job Satisfaction Level,...,Skills Gap,Family Influence Level,Has Mentor,Has Certification,Has Freelancing Experience,Willing to Relocate,Professional Network Strength,# Career Change Events,Technical Proficiency Level,Likely to Change Occupation
63003,784839,75617,Mechanical Engineering,Business Analyst,47.0,Male,1.0,High School,Medium,Low,...,Low,NaN,N,N,N,No,Average,1,Beginner,Likely
75474,281059,90827,Economics,Doctor,26.0,Female,34.0,Master's,Medium,High,...,High,Low,Y,N,N,No,Average,1,Intermediate,Likely
54896,235586,38672,Education,Artist,29.0,Male,0.0,Master's,Low,Moderate,...,Low,NaN,N,Y,N,No,Weak,1,Intermediate,Likely
828,297721,64190,Education,Artist,39.0,Male,32.0,High School,Medium,Moderate,...,Moderate,Medium,Y,N,N,No,Weak,2,Intermediate,Likely
88330,792307,90827,Law,Software Developer,37.0,Female,13.0,Master's,Medium,Moderate,...,High,High,N,Y,N,No,Weak,1,Beginner,Likely


In [51]:
# Count missing values in each column
total_null = raw_df.isnull().sum()
print(total_null)

Employee ID                          0
Employer ID                          0
Field of Study                       0
Current Occupation                   0
Age                                 35
Gender                             209
Years of Experience                486
Education Level                    304
Industry Growth Rate               599
Job Satisfaction Level             472
Work-Life Balance                  397
# Job Opportunities                390
Salary                             347
Job Security Level                   0
Interested in Career Change          0
Skills Gap                           0
Family Influence Level           25360
Has Mentor                         735
Has Certification                  633
Has Freelancing Experience           0
Willing to Relocate                  0
Professional Network Strength        0
# Career Change Events               0
Technical Proficiency Level          0
Likely to Change Occupation          0
dtype: int64


In [29]:
# Check for duplicate rows across all columns
total_duplicates = raw_df.duplicated().sum()
print(f'Dataset contains {total_duplicates} duplicate rows')

Dataset contains 253 duplicate rows


# 2) Data Removal

In [69]:
# Remove duplicate rows
df = raw_df.drop_duplicates()

In [70]:
# Confirm no duplicate rows remain
total_duplicates = df.duplicated().sum()
print(f'Dataset contains {total_duplicates} duplicate rows')

Dataset contains 0 duplicate rows


In [71]:
# Display updated DataFrame dimensions
df.shape  # 99,747 data instances (n), 25 features (m)

(99747, 25)

In [73]:
# Check if Employee ID is unique for every observation
df.shape[0] == df['Employee ID'].nunique()

True

In [74]:
# Set Employee ID as the index to uniquely identify each observation
df.set_index('Employee ID', inplace=True)

# Randomly inspect 5 observations
df.sample(5)

,Employer ID,Field of Study,Current Occupation,Age,Gender,Years of Experience,Education Level,Industry Growth Rate,Job Satisfaction Level,Work-Life Balance,...,Skills Gap,Family Influence Level,Has Mentor,Has Certification,Has Freelancing Experience,Willing to Relocate,Professional Network Strength,# Career Change Events,Technical Proficiency Level,Likely to Change Occupation
Employee ID,,,,,,,,,,,,,,,,,,,,,
488674,38672,Medicine,Lawyer,23.0,Female,39.0,Bachelor's,Low,Low,Poor,...,Low,High,N,Y,N,No,Strong,2,Intermediate,Likely
537122,92445,Psychology,Psychologist,32.0,Male,18.0,PhD,Medium,Low,Poor,...,High,Medium,Y,Y,N,No,Strong,1,Intermediate,Likely
616500,64190,Business,Business Analyst,35.0,Female,10.0,Master's,High,Moderate,Moderate,...,High,Medium,Y,Y,N,No,Average,2,Intermediate,Unlikely
546690,60225,Mechanical Engineering,Artist,45.0,Female,5.0,Bachelor's,High,Low,Poor,...,Low,Low,N,N,N,No,Weak,0,Intermediate,Likely
553648,53209,Psychology,Doctor,37.0,Male,37.0,High School,High,Low,Good,...,High,NaN,N,N,N,No,Average,1,Beginner,Likely


In [75]:
# Check for missing values
total_null = df.isnull().sum()
print(total_null)

Employer ID                          0
Field of Study                       0
Current Occupation                   0
Age                                 34
Gender                             205
Years of Experience                484
Education Level                    303
Industry Growth Rate               598
Job Satisfaction Level             471
Work-Life Balance                  397
# Job Opportunities                390
Salary                             345
Job Security Level                   0
Interested in Career Change          0
Skills Gap                           0
Family Influence Level           25293
Has Mentor                         730
Has Certification                  633
Has Freelancing Experience           0
Willing to Relocate                  0
Professional Network Strength        0
# Career Change Events               0
Technical Proficiency Level          0
Likely to Change Occupation          0
dtype: int64


In [78]:
df['Employer ID'].value_counts()

,count
Employer ID,
50154,6765
18718,6756
38672,6743
60225,6716
13309,6712
55281,6685
75617,6678
64190,6668
87326,6606


In [47]:
# Display DataFrame info: entries, columns, non-null counts, data types
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99747 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Employee ID                    99747 non-null  int64  
 1   Employer ID                    99747 non-null  int64  
 2   Field of Study                 99747 non-null  object 
 3   Current Occupation             99747 non-null  object 
 4   Age                            99713 non-null  float64
 5   Gender                         99542 non-null  object 
 6   Years of Experience            99263 non-null  float64
 7   Education Level                99444 non-null  object 
 8   Industry Growth Rate           99149 non-null  object 
 9   Job Satisfaction Level         99276 non-null  object 
 10  Work-Life Balance              99350 non-null  object 
 11  # Job Opportunities            99357 non-null  float64
 12  Salary                         99402 non-null  floa

In [76]:
# Generate summary statistics for numeric columns in transposed format
df.describe().T      # .T -> rotates the data 90 degrees

,count,mean,std,min,25%,50%,75%,max
Employer ID,99747.0,53543.333444,24529.951430,13309.0,38672.0,53209.0,75617.00,92445.0
Age,99713.0,39.602399,11.623597,20.0,30.0,40.0,50.00,59.0
Years of Experience,99263.0,19.485548,11.517518,0.0,10.0,20.0,29.00,39.0
# Job Opportunities,99357.0,50.287005,28.874180,1.0,25.0,50.0,75.00,100.0
Salary,99402.0,109017.152753,51212.112865,30005.0,61744.0,107696.0,153383.25,199996.0
# Career Change Events,99747.0,1.008923,0.814793,0.0,0.0,1.0,2.00,2.0


# 3) Handling Missing Values

(Natalie's work:)

In [ ]:
# Replace missing numeric values with an arithmetic (mean or median) strategy
#  Could also have used strategy='median'
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
df['Age'] = imp_mean.fit_transform(df[['Age']])
df['Years of Experience'] = imp_mean.fit_transform(df[['Years of Experience']])
df['Salary'] = imp_mean.fit_transform(df[['Salary']])
df['# Job Opportunities'] = imp_mean.fit_transform(df[['# Job Opportunities']])

In [ ]:
df['Age'].unique()

array([38.        , 44.        , 59.        , 57.        , 36.        ,
       43.        , 34.        , 20.        , 51.        , 55.        ,
       39.        , 45.        , 42.        , 21.        , 49.        ,
       32.        , 48.        , 23.        , 53.        , 47.        ,
       58.        , 56.        , 35.        , 25.        , 37.        ,
       27.        , 30.        , 31.        , 50.        , 54.        ,
       52.        , 46.        , 41.        , 24.        , 33.        ,
       40.        , 29.        , 26.        , 22.        , 28.        ,
       39.60266093])

In [ ]:
# Replace missing non-numeric values with a most_frequent (mode) strategy
imp_most_freq = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df['Gender'] = imp_most_freq.fit_transform(df[['Gender']]).ravel()
df['Education Level'] = imp_most_freq.fit_transform(df[['Education Level']]).ravel()
df['Job Satisfaction Level'] = imp_most_freq.fit_transform(df[['Job Satisfaction Level']]).ravel()
df['Has Mentor'] = imp_most_freq.fit_transform(df[['Has Mentor']]).ravel()
df['Has Certification'] = imp_most_freq.fit_transform(df[['Has Certification']]).ravel()
df['Industry Growth Rate'] = imp_most_freq.fit_transform(df[['Industry Growth Rate']]).ravel()
df['Work-Life Balance'] = imp_most_freq.fit_transform(df[['Work-Life Balance']]).ravel()